# L4c: Shortest-Path Algorithms

__Which route from a source to a target has the smallest total cost?__ Shortest-path algorithms answer this question by accounting for edge weights; the route with the fewest edges need not be the least expensive.

In [L4a](../L4a/CHEME-5800-L4a-Lecture-GraphAndTreeRepresentations-Fall-2026.ipynb), we compared graph representations, and in [L4b](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb), we used traversal algorithms to explore reachable vertices. We now use weighted graphs to compare the costs of reaching those vertices and reconstruct routes that minimize the total weight.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Formulate the shortest-path problem:__ Define route cost in a weighted directed graph and distinguish a finite shortest-path distance from an unreachable target. Use edge relaxation to update distance estimates and record predecessors for route reconstruction.
> * __Explain Dijkstra's algorithm:__ Trace how a priority queue selects vertices and how edge relaxation updates their neighbors. Explain why nonnegative edge weights allow selected distances to be declared final, and describe the algorithm's computational cost.
> * __Explain Bellman–Ford's algorithm:__ Trace repeated relaxation passes and explain why at most $|\mathcal{V}|-1$ passes suffice when no reachable negative-weight cycle exists. Distinguish negative edges from negative-weight cycles and explain how the algorithm detects a reachable negative cycle.

In this lecture, we develop the edge-relaxation rule shared by both algorithms, then compare their update strategies and assumptions. We use small directed graphs to examine when the algorithms agree and what changes when negative weights are present.

Let's get started!

___


## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines the lecture folder path, and loads the course package and its dependencies.

Let's set up our code environment:


In [ ]:
# Set up the lecture code and packages -
include(joinpath(@__DIR__, "Include.jl")); # activate the pinned course environment and load L4c imports

See the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5800 course package documentation](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/) for the functions and types used here.

The shortest-path implementations are in [`ShortestPathAlgorithms.jl`](../../../code/src/ShortestPathAlgorithms.jl). Dijkstra uses the [`PriorityQueue` type](https://juliacollections.github.io/DataStructures.jl/stable/priority-queue/) from [`DataStructures.jl`](https://juliacollections.github.io/DataStructures.jl/stable/); our checks use the [`Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/).

___



## Shortest-Path Problem and Relaxation

We focus on the __single-source shortest-path problem__: starting from one source vertex, find the minimum cost of reaching every other vertex. We first define a feasible route and its cost.

### Route Cost and Shortest-Path Distance

Let $\mathcal{G}=(\mathcal{V},\mathcal{E})$ be a finite directed graph with vertex set $\mathcal{V}$, edge set $\mathcal{E}$, and edge-weight function $w:\mathcal{E}\rightarrow\mathbb{R}$. We write $|\mathcal{V}|$ for the number of vertices and $|\mathcal{E}|$ for the number of edges. A route from source vertex $s$ to target vertex $t$ is a walk that follows the edge directions. We write a route containing $k$ edges as:
$$
P=\langle v_0,v_1,\ldots,v_k\rangle,\qquad v_0=s,\quad v_k=t,
$$
where $(v_i,v_{i+1})\in\mathcal{E}$ for $i=0,\ldots,k-1$. Its total cost is:
$$
w(P):=\sum_{i=0}^{k-1}w(v_i,v_{i+1}).
$$

__When does a minimum route cost exist?__ For a reachable target, a finite minimum exists unless an $s$-to-$t$ route contains a cycle whose edge weights sum to a negative number. Repeating that cycle keeps lowering the route cost. Thus, no minimum-cost route to that target exists.

> __Shortest-path distance:__
>
> Let $\mathcal{P}_{s,t}$ denote the set of feasible routes from $s$ to $t$. Provided that no negative-weight cycle lies on an $s$-to-$t$ route, the shortest-path distance is:
> $$
> d(s,t):=
> \begin{cases}
> \displaystyle\min_{P\in\mathcal{P}_{s,t}}w(P), & \mathcal{P}_{s,t}\neq\varnothing,\\[6pt]
> +\infty, & \mathcal{P}_{s,t}=\varnothing.
> \end{cases}
> $$
> The value $+\infty$ identifies an unreachable target. When a finite minimum exists, $\min$ gives the minimum route cost, while $\arg\min$ gives the set of routes that attain it. We use $P^*$ to denote one selected route from this set. The selected route and its cost satisfy:
> $$
> P^*\in\arg\min_{P\in\mathcal{P}_{s,t}}w(P),\qquad w(P^*)=d(s,t).
> $$

### Distance Estimates and Edge Relaxation

A single-source algorithm computes these distances for every target. During the calculation, $\operatorname{dist}[v]$ stores the best distance found so far; at successful completion, it equals $d(s,v)$. The predecessor $\operatorname{prev}[v]$ records the preceding vertex on the selected route. We use $\texttt{nothing}$ to mean that no predecessor is recorded. At successful completion, the source and any unreachable vertex have this value. For a reachable target, following predecessors backward to the source reconstructs the route. When routes tie, the algorithm may select any one of them.

__How can we avoid checking every route?__ Shortest paths have [__optimal substructure__](https://ocw.mit.edu/courses/6-006-introduction-to-algorithms-fall-2011/7a2abc9bc568c743404e85e85cf6dc59_MIT6_006F11_lec15.pdf#page=6): if a shortest path from $s$ to $t$ passes through $u$, its prefix from $s$ to $u$ must also be a shortest path. Otherwise, replacing that prefix with a cheaper one would produce a cheaper path to $t$.

This property motivates __edge relaxation__: we test whether reaching $v$ through $u$ gives a cheaper route than the best one found so far.

Initially, only the zero-edge route from $s$ to itself is known:
$$
\operatorname{dist}[v]=
\begin{cases}
0, & v=s,\\
+\infty, & v\ne s,
\end{cases}
\qquad
\operatorname{prev}[v]=\texttt{nothing}.
$$

If a finite route to $u$ has been discovered, appending the edge $(u,v)$ gives a candidate route to $v$. We use $\operatorname{alt}(u,v)$, short for __alternative route cost__, to denote the cost of this candidate route:
$$
\operatorname{alt}(u,v)=\operatorname{dist}[u]+w(u,v).
$$

This candidate may be cheaper than, equal to, or more expensive than the current estimate $\operatorname{dist}[v]$. In the pseudocode, $\operatorname{alt}$ stores this candidate cost for the edge $(u,v)$ currently being examined.

> __Edge relaxation:__
>
> If $\operatorname{alt}(u,v)<\operatorname{dist}[v]$, update the distance and predecessor:
> $$
> \operatorname{dist}[v]\leftarrow\operatorname{alt}(u,v),
> \qquad
> \operatorname{prev}[v]\leftarrow u.
> $$
> Otherwise, retain both values. The strict comparison preserves the existing predecessor when routes tie.

### Example: Relaxing One Edge

We will use the following four-vertex graph to compare the algorithms. Edge labels give the costs, and the red edges highlight the minimum-cost route from source vertex 1 to target vertex 4. We first examine how relaxation improves the cost of reaching vertex 2.

<div style="text-align: center;">
    <img src="figs/Fig-L4c-ShortestPathComparison.svg" width="560" alt="Directed graph with edges 1 to 2 of cost 4, 1 to 3 of cost 1, 3 to 2 of cost 2, 2 to 4 of cost 1, and 3 to 4 of cost 5. Red edges mark the minimum-cost route 1 to 3 to 2 to 4."/>
</div>

Suppose we start at vertex 1 and have discovered two direct routes: $1\rightarrow2$ with cost 4 and $1\rightarrow3$ with cost 1. Our current estimates are $\operatorname{dist}[2]=4$ and $\operatorname{dist}[3]=1$, and both vertices have predecessor 1.

Now suppose edge $(3,2)$ has weight 2. Can we reach vertex 2 more cheaply through vertex 3? The candidate route is $1\rightarrow3\rightarrow2$, with cost given by:
$$
\operatorname{alt}(3,2)=\operatorname{dist}[3]+w(3,2)=1+2=3.
$$
Because $3<4$, relaxing this edge improves the estimate for vertex 2 and changes its predecessor:
$$
\operatorname{dist}[2]\leftarrow3,
\qquad
\operatorname{prev}[2]\leftarrow3.
$$
Following the predecessors backward now takes us from vertex 2 to vertex 3 and then to the source, vertex 1. Reversing this sequence gives the route $1\rightarrow3\rightarrow2$. We have found a cheaper route by adding an edge to a route we already knew; we have not yet established that no further improvement is possible.

__Why does this help?__ For a target with a finite shortest-path distance, every finite estimate is the cost of a discovered route. It therefore cannot be smaller than the minimum route cost:
$$
d(s,v)\leq\operatorname{dist}[v].
$$
Relaxation lowers an estimate when we discover a cheaper route. Immediately after relaxing $(u,v)$, the estimates satisfy:
$$
\operatorname{dist}[v]\leq\operatorname{dist}[u]+w(u,v).
$$

Dijkstra's algorithm selects the smallest tentative distance and declares it final; nonnegative edge weights guarantee that later routes cannot improve it. Bellman–Ford repeatedly relaxes every edge, allowing later passes to improve earlier estimates even when some weights are negative.

___


## Dijkstra's Algorithm

Dijkstra's algorithm solves the single-source problem by repeatedly selecting the unprocessed vertex with the smallest tentative distance. Its greedy step is valid when every edge weight is nonnegative.

The __min-priority queue__ stores discovered vertices awaiting processing, with their current distance estimates as priorities. It returns the vertex with the smallest estimate and allows us to lower a queued vertex's priority when relaxation discovers a cheaper route.

### Algorithm

__Initialize:__ Given $\mathcal{G}=(\mathcal{V},\mathcal{E})$, source $s\in\mathcal{V}$, and weights $w(u,v)\geq0$, set $\operatorname{dist}[v]\leftarrow+\infty$ and $\operatorname{prev}[v]\leftarrow\texttt{nothing}$ for every vertex $v\in\mathcal{V}$. Set $\operatorname{dist}[s]\leftarrow0$, create an empty processed set $\mathcal{S}$ and an empty min-priority queue $\mathcal{Q}$, and insert $s$ into $\mathcal{Q}$ with priority zero.

While $\mathcal{Q}\neq\varnothing$ __do__:

1. __Select a vertex__: Remove a vertex $u$ with the smallest priority from $\mathcal{Q}$.
2. __Mark it as processed__: If $u\in\mathcal{S}$, continue to the next iteration of the while loop. Otherwise, set $\mathcal{S}\leftarrow\mathcal{S}\cup\{u\}$.
3. __Relax its outgoing edges__: For each edge $(u,v)\in\mathcal{E}$:
    1. Compute the alternative route cost to $v$ through $u$: $\operatorname{alt}\leftarrow\operatorname{dist}[u]+w(u,v)$.
    2. If $\operatorname{alt}<\operatorname{dist}[v]$, set $\operatorname{dist}[v]\leftarrow\operatorname{alt}$ and $\operatorname{prev}[v]\leftarrow u$. Insert $v$ into $\mathcal{Q}$ with priority $\operatorname{alt}$, or decrease its priority to $\operatorname{alt}$ if it is already in the queue.

__Return:__ When the queue is empty, return the distance map $\operatorname{dist}$ and predecessor map $\operatorname{prev}$.

### Example: Following the Distance Updates

Let's now trace Dijkstra's algorithm on the full four-vertex graph. In addition to the edges used in the preceding relaxation example, we include the edges into target vertex 4, with weights $w(2,4)=1$ and $w(3,4)=5$.

We start from vertex 1. The table records the distance estimates after each selected vertex's outgoing edges have been relaxed. The source estimate remains $\operatorname{dist}[1]=0$. Each queue entry is written as `vertex:priority`, and entries are listed in increasing priority order.

| Selected vertex | dist[2] | dist[3] | dist[4] | Queue after relaxation |
|:--|--:|--:|--:|:--|
| Initial state | ∞ | ∞ | ∞ | `1:0` |
| 1 | 4 | 1 | ∞ | `3:1`, `2:4` |
| 3 | 3 | 1 | 6 | `2:3`, `4:6` |
| 2 | 3 | 1 | 4 | `4:4` |
| 4 | 3 | 1 | 4 | Empty |

After processing vertex 1, we select vertex 3 because its estimate is 1. Relaxing $(3,2)$ lowers vertex 2's estimate from 4 to 3, as we calculated in the preceding example. Relaxing $(3,4)$ discovers a route to vertex 4 with cost $1+5=6$.

We next select vertex 2 with estimate 3. Relaxing $(2,4)$ lowers vertex 4's estimate from 6 to $3+1=4$ and changes its predecessor from 3 to 2. Vertex 4 is selected last. Following its predecessors backward takes us from vertex 4 to vertex 2, then vertex 3, and finally vertex 1. Reversing this sequence gives the route $1\rightarrow3\rightarrow2\rightarrow4$, with total cost 4.

Here, the selection order happens to match the reconstructed route. In general, we must use predecessors to recover a route. Each selected vertex's distance is final, while estimates for vertices still in the queue can improve. We now establish why nonnegative weights justify this distinction.

### Why the Greedy Step Is Correct

In our example, vertex 2 is selected with a tentative distance of __3__. Every other vertex waiting to be processed has a tentative distance of __at least 3__. Continuing through those vertices cannot produce a route costing less than 3, because each additional edge adds a nonnegative cost.

The proof's job is to establish that we have not overlooked a cheaper route to one of those waiting vertices. We prove this one vertex at a time, starting with the source: its distance is zero, and nonnegative edge weights prevent any route from having a lower cost.

> __Correctness of the greedy step:__
>
> Assume we have found the correct shortest-path distance for every previously selected vertex. Let $u$ be the next vertex selected, and suppose a cheaper path to $u$ exists.
>
> Along that path, let $y$ be the first vertex we have not yet selected and $x$ its predecessor. Because $x$ was already processed, relaxing $(x,y)$ gave $y$ an estimate no greater than the cost of the path prefix to $y$.
>
> The remaining edges have nonnegative weights, so that prefix cannot cost more than the full path to $u$. Thus, $y$ would have a smaller tentative distance than $u$, contradicting our choice of $u$. Therefore, the selected estimate is the shortest-path distance:
> $$
> \operatorname{dist}[u]=d(s,u).
> $$

__What fails with a negative edge?__ Suppose the edges $1\rightarrow2$, $1\rightarrow3$, and $3\rightarrow2$ have costs $2$, $5$, and $-4$, respectively. If we apply Dijkstra's selection rule from source vertex 1 despite the negative weight, we select vertex 2 with distance 2 before processing vertex 3. However, the route $1\rightarrow3\rightarrow2$ costs $5-4=1$. The negative edge therefore invalidates the claim that a selected distance is final.

Our implementation therefore rejects graphs containing negative edges.

### Computational Cost

With adjacency lists and a binary-heap priority queue, the running time is:
$$
\mathcal{O}\!\left((|\mathcal{V}|+|\mathcal{E}|)\log|\mathcal{V}|\right).
$$
Each reachable vertex is processed once, each outgoing edge is examined once, and queue insertions, priority updates, and minimum removals take logarithmic time.

___


## Bellman–Ford Algorithm

A negative edge weight can represent a credit in a cost model. For example, if a production step costs 1 unit but recovers reusable material worth 3 units, its net cost is $1-3=-2$.

Bellman–Ford allows negative edge weights by repeatedly relaxing every edge. Each pass uses the current distance estimates to look for cheaper routes. An improvement is available immediately to later edge relaxations in the same pass, so the order in which we visit the edges can affect how quickly the estimates converge.

### Algorithm

__Initialize:__ Given $\mathcal{G}=(\mathcal{V},\mathcal{E})$, source $s\in\mathcal{V}$, and weights $w(u,v)\in\mathbb{R}$, set $\operatorname{dist}[v]\leftarrow+\infty$ and $\operatorname{prev}[v]\leftarrow\texttt{nothing}$ for every vertex $v\in\mathcal{V}$. Then set $\operatorname{dist}[s]\leftarrow0$.

For each pass $k=1,2,\ldots,|\mathcal{V}|-1$ __do__:

1. __Reset the change flag__: Set $\texttt{changed}\leftarrow\texttt{false}$.
2. __Relax every edge__: For each edge $(u,v)\in\mathcal{E}$:
    1. If $\operatorname{dist}[u]=+\infty$, skip this edge and continue to the next edge.
    2. Compute the alternative route cost to $v$ through $u$: $\operatorname{alt}\leftarrow\operatorname{dist}[u]+w(u,v)$.
    3. If $\operatorname{alt}<\operatorname{dist}[v]$, set $\operatorname{dist}[v]\leftarrow\operatorname{alt}$, $\operatorname{prev}[v]\leftarrow u$, and $\texttt{changed}\leftarrow\texttt{true}$.
3. __Check for convergence__: If $\texttt{changed}=\texttt{false}$, stop the relaxation passes because no distance improved during the complete pass.

__Check for a negative-weight cycle:__ After the relaxation passes, visit every edge $(u,v)\in\mathcal{E}$ once more:

1. If $\operatorname{dist}[u]=+\infty$, skip this edge and continue to the next edge.
2. If $\operatorname{dist}[u]+w(u,v)<\operatorname{dist}[v]$, report a reachable negative-weight cycle and stop without returning shortest-path distances.

__Return:__ If the cycle check finds no further improvement, return the distance map $\operatorname{dist}$ and predecessor map $\operatorname{prev}$.

### Example: How Edge Order Affects the Passes

Let's use the same four-vertex graph as in the Dijkstra example. We visit its edges in the order used by the code in the worked comparison:
$$
(1,2),\quad(1,3),\quad(3,2),\quad(2,4),\quad(3,4).
$$
Starting from vertex 1, the first two relaxations set $\operatorname{dist}[2]=4$ and $\operatorname{dist}[3]=1$. Relaxing $(3,2)$ then lowers $\operatorname{dist}[2]$ to 3. Because $(2,4)$ comes next, it uses this improved value immediately and sets $\operatorname{dist}[4]=3+1=4$. The final edge offers cost $1+5=6$, so it makes no change.

All three distance estimates are correct after this first pass. A second pass makes no changes, and the algorithm stops the relaxation passes. What happens if we visit the same edges in reverse order? The table compares the estimates after each complete pass; each triple lists the distances to vertices 2, 3, and 4, in that order.

| Passes completed | Listed edge order | Reversed edge order |
|:--|:--|:--|
| 0 | (∞, ∞, ∞) | (∞, ∞, ∞) |
| 1 | (3, 1, 4) | (4, 1, ∞) |
| 2 | (3, 1, 4); no change, stop | (3, 1, 5) |
| 3 | Already stopped | (3, 1, 4) |

In the reversed order, we visit $(2,4)$ before $(3,2)$. When vertex 2's estimate improves during the second pass, its outgoing edge to vertex 4 has already been checked. That improvement reaches vertex 4 during the third pass. The edge order changes the number of passes needed, but both orders give the same shortest-path distances.

### Why the Pass Bound Holds

Consider a shortest path with three edges, assuming no negative-weight cycle is reachable from the source. By optimal substructure, each prefix is itself a shortest path. Relaxing its first edge establishes the correct distance to the first vertex after the source. Once that distance is correct, relaxing the second edge establishes the correct distance to the next vertex; the same reasoning applies to the last edge.

Because every edge is visited in every pass, these distances are correct after at most one, two, and three passes, respectively. Favorable edge orders can establish several of them in the same pass, as our example shows.

> __Number of relaxation passes:__
>
> Provided that no negative-weight cycle is reachable from the source, after $k$ passes every vertex with a shortest path containing at most $k$ edges has its correct shortest-path distance.
>
> To bound the number of passes, we can choose a shortest route that never repeats a vertex. Any repeated vertex creates a cycle, which we can remove without increasing the route's cost.
>
> Such a route visits at most $|\mathcal{V}|$ vertices and therefore contains at most $|\mathcal{V}|-1$ edges. Thus, $|\mathcal{V}|-1$ passes suffice to find every reachable vertex's shortest-path distance.

This is an upper bound that holds for every edge order. We can stop sooner when a complete pass makes no changes.

### Detecting a Reachable Negative-Weight Cycle

After $|\mathcal{V}|-1$ passes, check the edges once more. If no negative-weight cycle were reachable from the source, the pass bound tells us that every distance would already be correct. Thus, if an edge from a reachable vertex can still lower a distance estimate, a reachable negative-weight cycle exists.

If we can reach the destination through a negative-weight cycle, going around the cycle again always makes the route cheaper. Thus, there is no shortest path to that destination.

### Computational Cost

Each pass examines $|\mathcal{E}|$ edges, giving a worst-case running time of:
$$
\mathcal{O}(|\mathcal{V}|\,|\mathcal{E}|).
$$
The distance and predecessor maps require $\mathcal{O}(|\mathcal{V}|)$ storage. We can stop early when a complete pass makes no changes.

### Choosing a Shortest-Path Algorithm

When all edge weights are nonnegative, both algorithms return the same shortest-path distances. They may select different routes when several routes share the minimum cost.

| Feature | Dijkstra | Bellman–Ford |
|:--|:--|:--|
| Edge weights | Must be nonnegative | May be negative |
| Update strategy | Select the unprocessed vertex with the smallest distance estimate and relax its outgoing edges | Repeatedly relax every edge |
| Negative-cycle detection | Not supported | Detects negative-weight cycles reachable from the source |
| Worst-case time | $\mathcal{O}((\lvert\mathcal{V}\rvert+\lvert\mathcal{E}\rvert)\log\lvert\mathcal{V}\rvert)$ | $\mathcal{O}(\lvert\mathcal{V}\rvert\,\lvert\mathcal{E}\rvert)$ |
| Graph representation used here | Weighted adjacency list | Edge list |

Use Dijkstra when all edge weights are nonnegative. Use Bellman–Ford when negative edges may occur or when we need to detect a reachable negative-weight cycle.

___


## Worked Comparison

We now compute the routes and distances traced in the preceding sections, then examine what changes when negative weights are present.

### Comparing the Algorithms on a Nonnegative Graph

Our [four-vertex graph](#Example:-Relaxing-One-Edge) has three routes from vertex 1 to vertex 4. Adding their edge weights gives:

| Route | Total cost |
|:--|:--|
| 1 → 2 → 4 | 4 + 1 = 5 |
| 1 → 3 → 4 | 1 + 5 = 6 |
| 1 → 3 → 2 → 4 | 1 + 2 + 1 = 4 |

Both algorithms should return the same shortest-path distances. Here, they should also reconstruct the same route to vertex 4 because the minimum-cost route is unique.

We store the directed edges in `nonnegative_edges`, with each tuple specifying the starting vertex, ending vertex, and edge weight. We then select vertex 1 as the source and vertex 4 as the target for route reconstruction.


In [ ]:
# Build the nonnegative directed graph -
# Each record is (source vertex, target vertex, edge weight), in consistent cost units.
nonnegative_edges = weighted_edges([
    (1, 2, 4.0), # direct route from source 1 to vertex 2
    (1, 3, 1.0), # first step of the minimum-cost route
    (3, 2, 2.0), # reaches vertex 2 for cost 1 + 2 = 3, improving the direct cost 4
    (2, 4, 1.0), # completes the route to target 4 for total cost 4
    (3, 4, 5.0), # alternative route to target 4 has cost 1 + 5 = 6
])
source_vertex, target_vertex = 1, 4; # compute from source 1; reconstruct the route ending at target 4

[The `dijkstra(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.dijkstra-Tuple%7BAbstractVector%7BWeightedEdge%7D%2C%20Integer%7D) and [the `bellman_ford(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.bellman_ford-Tuple%7BAbstractVector%7BWeightedEdge%7D%2C%20Integer%7D) each compute the minimum cost of reaching every graph vertex from the source. Each result contains two maps: `distances` gives the minimum costs, and `previous` records the selected predecessors.

[The `reconstruct_path(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.reconstruct_path-Tuple%7BAbstractDict%2C%20Integer%2C%20Integer%7D) follows the predecessor map backward from the target to the source and reverses the sequence. We use it to recover the route to vertex 4 from each algorithm's result:


In [ ]:
# Compute distances and predecessors for every vertex from the same source -
dijkstra_result = dijkstra(nonnegative_edges, source_vertex)    # finalize distances in priority order
bellman_result = bellman_ford(nonnegative_edges, source_vertex) # improve distances over edge scans

# Follow each predecessor map backward, then reverse to obtain a source-to-target route -
shortest_path = reconstruct_path(dijkstra_result.previous, source_vertex, target_vertex) # Dijkstra route
bellman_path = reconstruct_path(bellman_result.previous, source_vertex, target_vertex);   # Bellman–Ford route

We now display the selected route and its cost. [The `path_cost(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.path_cost-Tuple%7BAbstractVector%7BWeightedEdge%7D%2C%20AbstractVector%7B%3C%3AInteger%7D%7D) adds the edge weights along that route. The two agreement checks compare every source-to-vertex distance and the reconstructed route to our target; both should be `true` for this graph.


In [ ]:
# Display the selected route, its cost, and agreement between the algorithms -
(
    path = shortest_path,                                                  # source-to-target vertex sequence
    cost = path_cost(nonnegative_edges, shortest_path),                     # sum the selected edge weights
    distances_agree = dijkstra_result.distances == bellman_result.distances, # compare costs for every vertex
    paths_agree = shortest_path == bellman_path,                            # this graph has a unique best route
)

The output confirms that both algorithms find $1\rightarrow3\rightarrow2\rightarrow4$, with total cost 4. This route uses more edges than either alternative but has the lowest cost. These results correspond to the edge weights above; changing a weight may change the selected route and its cost.

__What if every edge cost were 1?__ A route's total cost would equal its number of edges, so finding the shortest path would mean finding a route with the fewest edges. Breadth-first search, which we implemented in [L4b](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb), visits vertices in these minimum-edge-count layers. In this graph, the two-edge routes $1\rightarrow2\rightarrow4$ and $1\rightarrow3\rightarrow4$ would tie for the minimum cost of 2.


### Negative Edges Without Negative Cycles

A negative edge can belong to a shortest path. In the graph below, edge $(2,3)$ has weight $-2$, but the graph has no cycles. We use Bellman–Ford to compute the minimum-cost route from vertex 1 to vertex 4:


In [ ]:
# Build a graph with one negative edge and no cycles -
# The negative edge cannot be repeated, so it does not allow the route cost to fall without bound.
negative_edge_graph = weighted_edges([
    (1, 2, 4.0),  # first step of the minimum-cost route
    (1, 3, 5.0),  # direct alternative to vertex 3
    (2, 3, -2.0), # lowers the cost of reaching vertex 3 from 5 to 4 - 2 = 2
    (3, 4, 3.0),  # completes the route to target 4 for total cost 5
])
negative_result = bellman_ford(negative_edge_graph, 1)           # distances and predecessors from source 1
negative_path = reconstruct_path(negative_result.previous, 1, 4) # source-to-target vertex sequence

# Display the route and its total edge cost -
(
    path = negative_path,                                # route 1 → 2 → 3 → 4
    cost = path_cost(negative_edge_graph, negative_path), # total cost 4 - 2 + 3 = 5
)

Bellman–Ford returns the route $1\rightarrow2\rightarrow3\rightarrow4$, with total cost $4-2+3=5$. The negative edge lowers the route's cost, while the absence of cycles ensures that we cannot keep lowering it by repeating part of the route. This result corresponds to the edge weights above; changing them may change the minimum-cost route.

What happens if we give the same graph to Dijkstra? Our implementation requires nonnegative edge weights, so it rejects this input. We capture the expected error message so that we can inspect it and continue running the notebook:


In [ ]:
# Capture Dijkstra's expected rejection of a negative edge -
dijkstra_rejection = try
    dijkstra(negative_edge_graph, 1) # rejects the -2 edge before processing vertices
    "unexpectedly accepted"         # returned only if the input check fails to reject the graph
catch error
    sprint(showerror, error)         # display the diagnostic while allowing later cells to run
end

The message confirms that Dijkstra rejects the negative edge. Bellman–Ford's result shows that this graph still has a finite minimum-cost route.

### A Reachable Negative-Weight Cycle

Now consider a graph containing the cycle $1\rightarrow2\rightarrow3\rightarrow1$, with total weight $1-2+0=-1$. Each trip around the cycle reduces the route cost by 1. Starting from vertex 1, we can therefore keep lowering the cost of reaching any vertex in this graph, so no shortest path exists.

Bellman–Ford detects the reachable negative-weight cycle and reports an error. We capture its message in the same way:


In [ ]:
# Build a reachable cycle with total weight 1 - 2 + 0 = -1 -
negative_cycle_graph = weighted_edges([
    (1, 2, 1.0),  # leave the source with cost 1
    (2, 3, -2.0), # reduce the running cost to -1
    (3, 1, 0.0),  # return to source 1; each repetition reduces the cost by another 1
])

# Capture Bellman–Ford's expected rejection of the reachable negative-weight cycle -
cycle_rejection = try
    bellman_ford(negative_cycle_graph, 1) # a final edge scan detects a further possible improvement
    "unexpectedly accepted"              # returned only if cycle detection fails to reject the graph
catch error
    sprint(showerror, error)              # display the diagnostic while allowing later cells to run
end

The message confirms that Bellman–Ford detected the reachable negative-weight cycle. The first example produced a finite shortest path containing a negative edge; the second allowed the route cost to decrease without bound by repeating a cycle.

### Checking the Results

The checks below verify the expected routes and costs, agreement between the algorithms on the nonnegative graph, and both expected error conditions. If you change the graph's weights or connections, revisit the expected results before rerunning these checks.


In [ ]:
# Check route costs, algorithm agreement, and the two expected error conditions -
@testset "shortest-path contracts" begin
    # Nonnegative graph: both algorithms find the unique minimum-cost route -
    @test shortest_path == [1, 3, 2, 4]                         # three edges, rather than the two-edge alternatives
    @test dijkstra_result.distances[4] == 4.0                   # target cost is 1 + 2 + 1
    @test dijkstra_result.distances == bellman_result.distances # compare every source-to-vertex cost
    @test shortest_path == bellman_path                        # the unique best route also agrees

    # Negative edge: Bellman–Ford returns a finite minimum cost -
    @test negative_path == [1, 2, 3, 4]                         # the -2 edge belongs to the selected route
    @test negative_result.distances[4] == 5.0                   # target cost is 4 - 2 + 3

    # Confirm the error types, beyond displaying their messages in earlier cells -
    @test_throws ArgumentError dijkstra(negative_edge_graph, 1)     # negative edges violate Dijkstra's assumption
    @test_throws ArgumentError bellman_ford(negative_cycle_graph, 1) # a reachable negative cycle prevents a finite result
end

___

## Lab Exercises
In [L4d](../L4d/CHEME-5800-L4d-Lab-ProductionPlanningShortestPath-Fall-2026.ipynb), we compare two production routes: one uses fewer steps, while the other costs less. We calculate both route costs by hand, find the least-cost route with Dijkstra, and verify the result with Bellman–Ford.

___


## Summary
We developed two algorithms for finding minimum-cost routes and examined how edge weights determine which algorithm we can use.

> __Key Takeaways:__
>
> * **Route costs and relaxation:** We formulated the shortest-path problem by adding edge weights along a route. We used relaxation to improve distance estimates and recorded predecessors to reconstruct the selected route. When every edge costs 1, minimizing cost reduces to finding the fewest edges—the problem solved by breadth-first search.
>
> * **Dijkstra's algorithm:** We used a priority queue to select the unprocessed vertex with the smallest distance estimate. We showed why nonnegative edge weights make that estimate the correct shortest-path distance, and compared the result with Bellman–Ford on a graph with three competing routes.
>
> * **Bellman–Ford's algorithm:** We used repeated relaxation passes to find shortest paths when negative edges are present. We explained why $|\mathcal{V}|-1$ passes suffice when no negative-weight cycle is reachable from the source, and how an additional pass detects such a cycle. Our examples distinguished a negative edge that lowers a route's cost from a negative-weight cycle that prevents a shortest path from existing.

In [the L4d lab](../L4d/CHEME-5800-L4d-Lab-ProductionPlanningShortestPath-Fall-2026.ipynb), we apply these ideas to a production network, where the route with the fewest steps may not be the least expensive.

___
